In [ ]:
import malariagen_data

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:
%load_ext line_profiler
%load_ext memory_profiler

In [ ]:
af1 = malariagen_data.Af1()
af1

## Local tabix-indexed VCF in IGV
This exports one sample to a BGZF VCF and `.tbi` under the notebook working directory, then loads it as the SNP track. Both files must be inside the Jupyter server startup directory tree. The `snp_vcf_url` below assumes the server was started from the workspace root; if it was started from `notebooks/`, use the alternate URL shown in the code cell.

In [ ]:
from pathlib import Path

#%%capture cap --no-stderr

sample_set = "1354-VO-KE-DONNELLY-VMF00281"
sample_id = str(
    af1.sample_metadata(sample_sets=sample_set).iloc[0]["sample_id"]
)
region = "2RL:2,440,000-2,445,000"

local_vcf_path = Path("local_igv_vcfs") / f"{sample_id}.vcf.gz"
local_vcf_path.parent.mkdir(parents=True, exist_ok=True)
regions2 = ["2RL:2,440,000-2,450,000", "2RL:2,440,000-2,460,000", "2RL:2,440,000-2,470,000", "2RL:2,440,000-2,480,000",
           "2RL:2,440,000-2,490,000", "2RL:2,440,000-2,500,000", "2RL:2,440,000-2,510,000", "2RL:2,440,000-2,520,000", 
            "2RL:2,440,000-2,530,000", "2RL:2,440,000-2,540,000"]

local_vcf_path_distinguishers = [str(i*10000) for i in range(10)]
def run_snp_calls(path_distinguisher, region):
    local_vcf_path = Path("local_igv_vcfs", f"{sample_id}_{path_distinguisher}.vcf.gz")
    #local_vcf_path.parent.mkdir(parents=True, exist_ok=True),
    af1.snp_calls_to_vcf(
        output_path=str(local_vcf_path),
        region=region,
        sample_sets=[sample_set],
        sample_query=f"sample_id == '{sample_id}'",
        overwrite=True,
    )


In [ ]:
for path_distinguisher, region in zip(local_vcf_path_distinguishers, regions2):
    print(f"Memory usage for region {region}")
    %memit run_snp_calls(path_distinguisher, region)


In [ ]:
def view_alignments(output_path, region):
    af1.view_alignments(
        sample=sample_id,
        region=region,
        snp_vcf_path=output_path,
    )


In [ ]:
for path_distinguisher, region in zip(local_vcf_path_distinguishers, regions2):
    local_vcf_path = Path("local_igv_vcfs", f"{sample_id}_{path_distinguisher}.vcf.gz")
    local_vcf_path.parent.mkdir(parents=True, exist_ok=True),
    output_path=str(local_vcf_path)
    print(f"Memory usage for region {region}")
    %memit view_alignments(output_path, region)
